# Direct String Matching and Result Contracts


```{contents}
:local:
:depth: 2
```


```{index} Define What Counts as a Match Before Choosing an Algorithm
```



```{index} Separate a Candidate Start from Its Matched Length
```

## Separate a Candidate Start from Its Matched Length


A direct matcher asks the same small question at each legal start: do all pattern code units agree here? There are two positions to track. Start chooses a candidate in the text; offset chooses a position inside the pattern. A mismatch rejects the current candidate only. The next start may still match, including an occurrence overlapping the previous match.

For text ABABA and pattern ABA:

| Candidate start | Character comparisons | Decision |
| --- | --- | --- |
| 0 | A=A, B=B, A=A | Report 0 |
| 1 | B differs from A | Reject only this start |
| 2 | A=A, B=B, A=A | Report 2 |

The final legal nonempty start is text.Length-pattern.Length. A longer pattern has no legal start. The empty-pattern policy is separate: this book reports every boundary, including the one after the final text code unit. The focused example below assumes non-null strings; the retained full API also rejects null inputs explicitly.


In [1]:
using System;
using System.Collections.Generic;
static int[] Direct(string text, string pattern)
{
    var starts = new List<int>();
    for (int start = 0; start <= text.Length - pattern.Length; start++)
    {
        int offset = 0;
        while (offset < pattern.Length && text[start + offset] == pattern[offset]) offset++;
        if (offset == pattern.Length) starts.Add(start);
    }
    return starts.ToArray();
}
Console.WriteLine($"Overlaps: {string.Join(",", Direct("ABABA", "ABA"))}");
Console.WriteLine($"Longer count: {Direct("AB", "ABC").Length}");
Console.WriteLine($"Empty starts: {string.Join(",", Direct("AB", ""))}");


Overlaps: 0,2
Longer count: 0
Empty starts: 0,1,2


Offset equals pattern.Length only when the whole pattern has matched. On a mismatch it stops earlier, so the candidate is not reported. Increasing start by one preserves overlaps. For the empty pattern, the inner loop runs zero times, offset already equals zero, and starts zero through text.Length are reported under the stated boundary policy.

Guided task: trace AA in AAAA and fill the starts without running code. Then change the outer update to advance by pattern.Length after a match. Which required starts disappear, and why is the modification incorrect for this contract?

```{dropdown} Check the overlap policy
The required starts are zero, one and two. Skipping two positions after a match loses start one. A non-overlapping API could choose that policy, but it would be solving a different problem. Tests must compare equivalent contracts.
```

For independent comparison with KMP, check empty text, empty pattern, longer patterns, repeated symbols, absent patterns and case differences before counting work. These examples compare ordinal UTF-16 code units. They do not normalize text or implement a culture-aware search. The retained hop-versus-cost discussion below is an objective-contract comparison linked to the graph chapters, not a prerequisite for the string algorithm.


```{index} Define What Counts as a Match Before Choosing an Algorithm
```

## Define What Counts as a Match Before Choosing an Algorithm



A message filter searches text for a literal pattern. Decide whether matching is case-sensitive, whether all overlapping starts are required, and how an empty pattern is treated. These choices determine the correct result before any comparison-count claim can be made. The core examples compare ordinal UTF-16 code units; a returned position is an offset in that representation, not necessarily a user-perceived character count.

For text ABABA and pattern ABA, the starts are zero and two. Moving the candidate start by the pattern length after the first match would miss the overlapping occurrence at two. A direct matcher instead considers every candidate start whose remaining text is long enough, comparing pattern elements until a mismatch or a complete match.

| Candidate start | Comparison outcome | Report it? |
| --- | --- | --- |
| 0 | ABA matches ABA | Yes |
| 1 | B differs from A immediately | No |
| 2 | ABA matches ABA | Yes |

The direct algorithm's simplicity makes it a useful independent reference for more elaborate matching. Its contract must still handle an empty pattern. Under the all-boundaries convention used here, an empty pattern matches before the first code unit, between adjacent code units, and after the final code unit. Thus text of length n has n+1 empty-pattern starts. Another API can choose another contract, but comparisons must use the same one.

For related explanation, see the further reading below.[^development-reading]


```{index} Use Mismatches as Evidence
```



```{index} Use Mismatches as Evidence
```

## Use Mismatches as Evidence



A mismatch at one candidate start establishes only that this candidate fails. It does not prove that the text contains no later occurrence. The outer loop advances the start; the inner loop advances the pattern position. Confusing those two positions can skip starts or compare the wrong slices.

With text length n and pattern length m, a direct comparison model can perform up to proportional to n*m work for nonempty patterns within the text. A long shared prefix followed by a mismatch can repeat work at many starts. The next lesson develops how prefix information can avoid some repetition. For short patterns or a small number of queries, a clear direct approach or a suitable library operation can still be a reasonable engineering choice; a more sophisticated algorithm is not required merely because it exists.

Guided check: find all starts of AA in AAAA, then an absent pattern longer than the text, then the empty pattern. Derive expected outputs manually. Use those cases when checking the preserved implementations so that a fast candidate cannot quietly change overlap or boundary semantics. Keep weighted-graph hop/cost reminders as links to Chapters 20 and 24; they are separate result-contract examples, not prerequisites for reading a string matcher.


```{index} Direct Matching with an Explicit Contract
```

## Direct Matching with an Explicit Contract




```{index} Direct Matching with an Explicit Contract
```

Search exact ordinal UTF-16 code units, returning every start position, including overlapping matches. A missing nonempty pattern returns an empty array. An empty pattern matches each boundary 0..text.Length; null arguments are rejected. Derived first-match clients return the first position or −1. Counts include every executed code-unit equality test, including mismatches.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

foreach(var pair in new[]{("aaaaa","aaa"),("abracadabra","cad"),("abc",""),("","x")})
{
 var result=PatternScan.Direct(pair.Item1,pair.Item2);
 Console.WriteLine($"text=[{pair.Item1}], pattern=[{pair.Item2}], positions=[{string.Join(",",result.Positions)}], comparisons={result.Comparisons}");
}
public sealed class PatternScan
{
    public record Matches(int[] Positions,long Comparisons);
    private readonly string pattern;
    private readonly int[] prefix;
    public long PreparationComparisons { get; }
    public int[] Prefix => (int[])prefix.Clone();
    public PatternScan(string pattern)
    {
        ArgumentNullException.ThrowIfNull(pattern);this.pattern=pattern;
        prefix=new int[pattern.Length];long comparisons=0;int matched=0;
        for(int i=1;i<pattern.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(pattern[i]==pattern[matched]){prefix[i]=++matched;break;}
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        PreparationComparisons=comparisons;
    }
    // Ordinal UTF-16 code units; all overlapping matches, including empty boundaries.
    public Matches Search(string text)
    {
        ArgumentNullException.ThrowIfNull(text);
        if(pattern.Length==0)return new Matches(Enumerable.Range(0,text.Length+1).ToArray(),0);
        var positions=new List<int>();int matched=0;long comparisons=0;
        for(int i=0;i<text.Length;i++)
        {
            while(true)
            {
                comparisons++;
                if(text[i]==pattern[matched])
                {
                    matched++;
                    if(matched==pattern.Length){positions.Add(i-pattern.Length+1);matched=prefix[matched-1];}
                    break;
                }
                if(matched==0)break;
                matched=prefix[matched-1];
            }
        }
        return new Matches(positions.ToArray(),comparisons);
    }
    public static Matches Direct(string text,string pattern)
    {
        ArgumentNullException.ThrowIfNull(text);ArgumentNullException.ThrowIfNull(pattern);
        var positions=new List<int>();long comparisons=0;
        for(int start=0;start<=text.Length-pattern.Length;start++)
        {
            int offset=0;
            while(offset<pattern.Length)
            {comparisons++;if(text[start+offset]!=pattern[offset])break;offset++;}
            if(offset==pattern.Length)positions.Add(start);
        }
        return new Matches(positions.ToArray(),comparisons);
    }
}


text=[aaaaa], pattern=[aaa], positions=[0,1,2], comparisons=9
text=[abracadabra], pattern=[cad], positions=[4], comparisons=11
text=[abc], pattern=[], positions=[0,1,2,3], comparisons=0
text=[], pattern=[x], positions=[], comparisons=0


The loop tests each legal start once. At offset j, the first j code units match; a mismatch rules out that start, while j=m proves an occurrence. For n text units and m>0 pattern units, worst-case comparison work is O((n−m+1)m) when m≤n, commonly written O(nm); m>n gives zero tests. Returned r positions cost O(r) storage. Empty patterns require O(n) output despite zero equality tests.

**Common mistake.** Advancing the start position by the pattern length after a match. That skips overlapping occurrences, so `aaa` in `aaaaa` reports one position instead of three; the contract says every legal start is tested, so the start advances by one.


The distinction between minimum hops and minimum cost is developed with its full graph counterexample in {doc}`Shortest Paths and Dijkstra <../24_greedy_graph_optimization/2404_shortest_paths>`. String matching likewise begins by choosing an explicit result contract.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Sedgewick and Wayne, Strings](https://algs4.cs.princeton.edu/50strings/). The worked business scenarios and guided reasoning here are original teaching examples.
